# 実行テスト

## 4.2 Normalizing activatins with layer normalization

In [1]:
!python layer_norm.py

 Layer Normalization Test 
入力シェイプ: torch.Size([2, 3, 4])
出力シェイプ: torch.Size([2, 3, 4])

[正規化後の確認]
出力の平均 (dim=-1): tensor([[2.9802e-08, 0.0000e+00, 0.0000e+00],
        [7.4506e-09, 0.0000e+00, 1.4901e-08]], grad_fn=<MeanBackward1>)
出力の分散 (dim=-1): tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 0.9999]], grad_fn=<VarBackward0>)


## 4.3 Implementing a feed forward network with GELU activation

In [2]:
!python gelu.py

In [3]:
!python feed_forward.py

FeedForward のテスト
入力テンソルの形状: torch.Size([2, 3, 768])
出力テンソルの形状: torch.Size([2, 3, 768])
入力と出力の形状が一致していればOK -> 一致しているか：True


## 4.4 Adding shortcut connection

In [4]:
!python deep_neural_network.py

ショートカット接続なしのモデル
layers.0.0.weight... gradient mean: 0.000202
layers.1.0.weight... gradient mean: 0.000120
layers.2.0.weight... gradient mean: 0.000715
layers.3.0.weight... gradient mean: 0.001399
layers.4.0.weight... gradient mean: 0.005050

ショートカット接続あり
layers.0.0.weight... gradient mean: 0.221698
layers.1.0.weight... gradient mean: 0.206941
layers.2.0.weight... gradient mean: 0.328970
layers.3.0.weight... gradient mean: 0.266573
layers.4.0.weight... gradient mean: 1.325854


## 4.5 Connecting attention and linear layers in a transfomer block

In [5]:
!python transformer_block.py

Transformer Block 実行結果
入力シェイプ: torch.Size([2, 4, 768])
出力シェイプ: torch.Size([2, 4, 768]) # (バッチサイズ、シーケンス長、埋め込み次元)
入力と出力の形状が一致

出力テンソル
tensor([[[-0.0055,  0.0972, -0.1122,  ...,  1.2889,  0.2623,  0.6685],
         [ 0.0023, -0.2369,  0.1720,  ...,  0.5952,  0.2497,  0.7447],
         [ 0.4673,  0.4472,  0.1791,  ...,  1.2525,  0.3045,  0.7750],
         [ 0.0662,  0.7224,  0.9206,  ...,  0.4790,  0.7428,  0.7015]],

        [[ 0.3622,  1.2144,  0.5221,  ...,  0.1854,  0.0111, -0.5034],
         [-0.0225,  0.7789,  0.2770,  ...,  0.1734,  0.5419,  0.1143],
         [ 0.7425,  0.4013,  0.3211,  ...,  0.3268,  0.7523, -0.1642],
         [ 0.5745,  0.6241,  0.4410,  ...,  1.1963,  1.2650,  0.2243]]],
       grad_fn=<AddBackward0>)


## 4.6 Cording the GPT model

In [6]:
!python gpt_model.py

GPT Model 実行結果
入力バッチのシェイプ: torch.Size([2, 4]) # (バッチサイズ、シーケンス長)
出力ロジットのシェイプ: torch.Size([2, 4, 50257]) # (バッチサイズ、シーケンス長、語彙数)

出力テンソル
tensor([[[-0.1378, -0.8384, -0.1209,  ...,  0.7960,  0.2566, -0.5706],
         [ 0.1667, -0.5663,  0.4006,  ..., -0.6424,  0.4922, -0.1980],
         [ 1.9094, -0.7840, -0.5078,  ...,  0.3276, -0.1901, -0.7636],
         [-0.3718, -0.3764,  0.2535,  ...,  0.3980,  0.3020, -0.5051]],

        [[-0.5684,  0.8721,  0.7665,  ...,  0.0150,  0.6408, -1.2515],
         [-0.0743,  0.1229,  0.2112,  ..., -0.0653,  0.1146,  0.4727],
         [ 0.7593,  0.2936,  0.3338,  ...,  0.2375, -0.3661,  1.1797],
         [-0.9017,  0.0195,  0.9662,  ...,  0.1955, -0.4636,  0.1075]]],
       grad_fn=<UnsafeViewBackward0>)

パラメータ数の確認
Total parameters in the model: 163,009,536
Token embedding layer shape: torch.Size([50257, 768])
Output head layer shape: torch.Size([50257, 768])
Total parameters in GPT-2 (excluding output head): 124,412,160


## 4.7 Generating text

In [7]:
import torch
import tiktoken

from gpt_model import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257, 
    "emb_dim": 768, 
    "context_length": 1024, 
    "drop_rate": 0.1, 
    "n_layers": 12, 
    "n_heads": 12, 
    "qkv_bias": False,
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval() 

tokenizer = tiktoken.get_encoding("gpt2")

def generate_text_simple(model, idx, max_new_token, context_size):
    for _ in range(max_new_token):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad(): 
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [8]:
start_context = "Hello, I am"

# テキストをトークンIDにエンコードし、[1, N] のPyTorchテンソルにする
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) 

print("処理開始")
print("入力テンソル:", encoded_tensor)

# テキスト生成を実行（10トークン追加）
out = generate_text_simple(
    model=model, 
    idx=encoded_tensor, 
    max_new_token=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("出力テンソル:", out)

# テンソルをリストに戻してテキストにデコード
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print("\n生成結果")
print(decoded_text)

処理開始
入力テンソル: tensor([[15496,    11,   314,   716]])
出力テンソル: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])

生成結果
Hello, I am Featureiman Byeswickattribute argue
